# **Maestría en Inteligencia Artificial Aplicada**

## Curso: **Procesamiento de Lenguaje Natural**

### Tecnológico de Monterrey

### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipo - Semanas 4 y 5**

### **Vectores Embebidos de HuggingFace**

#### **Nombres y matrículas de los integrantes del equipo:**

##### **Equipo 20**


- Gabriela del Carmen González Domínguez - A01796282
- Bertha Itzel Salamanca Murcia - A01797439
- Omar Aguilar Macedo - A01797078



In [1]:
#@title Instalar Librerías Extra

%pip install contractions

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.8 MB/s eta 0:00:00


In [2]:
#@title Agregar Librerías

# Aquí deberán incluir todas las librerías que requieran durante esta actividad:

import pandas as pd
import numpy as np

import csv
import nltk
from nltk.corpus import stopwords

import re
import contractions
import unicodedata
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.model_selection import train_test_split
from collections import Counter

from sentence_transformers import SentenceTransformer, util

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

import pickle

In [3]:
#@title Descargar diccionarios
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

En esta actividad deberás utilizar los datos de tres archivos que se encuentran en el repositorio de la UCI llamados **amazon_cells_labelled.txt**, **imdb_labelled.txt** y   **yelp_labelled.txt**. Cada uno de estos archivos corresponden a comentarios de usuarios que adquirieron un celular a través de la plataforma de Amazon, de comentarios que dejaron usuarios sobre palículas y series en la plataforma de IMDb y sobre servicios de comida dejados en la plataforma de Yelp.

La información del problema y de los archivos están basados en el repositorio de la UCI cuya liga es la siguiente:

https://archive.ics.uci.edu/dataset/331/sentiment+labelled+sentences



# **Pregunta - 1:**



Descarga los 3 archivos de la plataforma de la UCI indicado previamente y genera un nuevo DataFrame de Pandas con ellos.

**Llama simplemente "df" a dicho DataFrame.**




In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


dfa = pd.read_csv(
    '/content/drive/MyDrive/NLP/sentiment labelled sentences/amazon_cells_labelled.txt', sep='\t',
    names=['review','label'], header=None,
    encoding='utf-8'
)
dfi = pd.read_csv(
    '/content/drive/MyDrive/NLP/sentiment labelled sentences/imdb_labelled.txt', delimiter='\t',
    names=['review','label'], header=None,
    quoting=csv.QUOTE_NONE,
    encoding='utf-8'
)
dfy = pd.read_csv(
    '/content/drive/MyDrive/NLP/sentiment labelled sentences/yelp_labelled.txt', sep='\t',
    names=['review','label'], header=None,
    encoding='utf-8'
)

print('Total de registros de Amazon:', dfa.shape, type(dfa))
print('Total de registros de IMBD:', dfi.shape, type(dfi))
print('Total de registros de Yelp:', dfy.shape, type(dfy))


df = pd.concat([dfa, dfi, dfy], ignore_index=True)


# *********** Aquí termina la sección de agregar código *************

Total de registros de Amazon: (1000, 2) <class 'pandas.core.frame.DataFrame'>
Total de registros de IMBD: (1000, 2) <class 'pandas.core.frame.DataFrame'>
Total de registros de Yelp: (1000, 2) <class 'pandas.core.frame.DataFrame'>


In [10]:
# Verifiquemos la información del DataFrame:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   review  3000 non-null   object
 1   label   3000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 47.0+ KB


In [11]:
# Y mostremos sus primeros registros:

df.head()

,review,label
0,So there is no way for me to plug it in here i...,0
1,"Good case, Excellent value.",1
2,Great for the jawbone.,1
3,Tied to charger for conversations lasting more...,0
4,The mic is great.,1


# **Pregunta - 2:**

Proceso de limpieza. Aplica el proceso de limpieza que consideres adecuado.











In [12]:
# Consideremos la siguiente lista de palabras asociada a negaciones en inglés:
negwords = [ 'no', 'nor', 'not', 'ain', 'aren', "aren't", 'don', "don't", 'couldn', "couldn't", 'didn', "didn't", 'doesn', "doesn't", 'hadn', "hadn't", 'hasn', "hasn't", 'haven', "haven't", 'isn', "isn't", 'mightn', "mightn't", 'mustn', "mustn't", 'needn', "needn't", 'shan', "shan't", 'shouldn', "shouldn't", 'wasn', "wasn't", 'weren', "weren't", 'won', "won't", 'wouldn', "wouldn't"]
# negwords = {'no', 'not', 'nor', 'neither', 'never', 'none', 'isn', "isn't", 'aren', "aren't", 'wasn', "wasn't", 'weren', "weren't", 'hasn', "hasn't", 'haven', "haven't", 'hadn', "hadn't", 'doesn', "doesn't", 'don', "don't", 'didn', "didn't", 'won', "won't", 'wouldn', "wouldn't", 'shouldn', "shouldn't", 'can', "can't", 'couldn', "couldn't", 'mustn', "mustn't", 'ma', 'mightn', "mightn't"}

# Y las excluimos de las stopwords:
mystopwords = [ w for w in stopwords.words('english') if w not in negwords]


print("Total de stopwords en la lista original de NLTK: %d" % len(stopwords.words('english')))
print("Total de sotpwords excluyendo los conectivos negativos: %d\n" % len(mystopwords))

# Convertimos la lista de stopwords a `set` por un poco de eficiencia.
stopwords_set = set(mystopwords)


# Nuestra nueva lista:
print(stopwords_set)

Total de stopwords en la lista original de NLTK: 198
Total de sotpwords excluyendo los conectivos negativos: 158

{'each', 'the', 'we', "he's", 'his', 'be', 'most', 'y', 'how', 'when', "he'd", 'against', 'of', 'ours', 'has', 'then', "i've", "we're", "she'll", 'i', 'over', 'their', 'them', 'again', 'under', 've', 'with', 'all', 'what', 'very', 'my', 'off', "they'd", "she's", 'are', 'if', 'too', 'he', 'just', 'yourself', "they're", 'her', 'had', 'd', 'where', 'theirs', 'you', 'now', 's', "they'll", 'doing', 'few', 'ourselves', 'before', 'than', 'a', 'between', "you've", 'any', 'they', 'whom', 'o', 'while', 'yours', "that'll", "you'd", 'that', 'about', 'on', 'own', 'were', 'down', 'hers', 'himself', "it'll", 'this', 'itself', 'which', 'who', 'and', "should've", "we'd", 'am', 'so', 'can', 'those', "she'd", "we'll", "i'm", 'why', 'our', 'until', 'only', 'because', 'through', 'further', 'by', 'in', 'more', 'other', 'she', "i'd", 'is', "you're", 'ma', 'for', 'into', 'm', 'or', 'out', 'some', 

In [13]:
 # Ahora separemos la información:
 #     La "X" serán los datos de entrada, los comentarios.
 #     La "Y" será la variable de salida, las etiquetas de la evaluación.
 # Ambos, X y Y son "Series"

X = df.review     # Serie de strings
y = df.label      # Serie de enteros 0s y 1s

assert X.shape == (3000,)           # Verificando que tenemos la dimensiones esperadas.
assert y.shape == (3000,)           # Si todo va bien, no debe mostrarse algún tipo de error al ejectuar esta celda.

In [14]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

def get_root_base_stem(token):
  return stemmer.stem(token)

def get_root_base_lemma(token):
  new_token = lemmatizer.lemmatize(token, pos='v') # verbos
  new_token = lemmatizer.lemmatize(new_token, pos='n') # sustantivos
  new_token = lemmatizer.lemmatize(new_token, pos='a') # adjetivos
  return new_token

def get_root_base(token):
  return get_root_base_lemma(token);

def normalize_repeated_chars(token):
  # baaaaaaad -> baad
  # goooood -> good
  # zoo -> zoo
  return re.sub(r'(.)\1{2,}', r'\1\1', token)

def quitar_acentos(texto):
    # Descompone caracteres especiales en su letra base + el acento
    texto_normalizado = unicodedata.normalize('NFKD', texto)
    # Nos quedamos solo con los caracteres que NO son acento
    texto_sin_acentos = ''.join([
        c for c in texto_normalizado if not unicodedata.combining(c)
    ])
    return texto_sin_acentos

def transformar_contracciones(texto):
  return contractions.fix(texto)

## Ejemplo de como podríamos normalizar calificaciones
##  es un método ingenuo ya que asume que son califiaciones con cierto formato
def normalizar_calificaciones(texto):
    # Detectar 10/10, 9/10 y convertirlos en 'excelente'
    texto = re.sub(r'\b(?:9|10)/10\b', ' excelent ', texto)

    # Detectar notas medias de 5/10 a 8/10 y convertirlos en 'bueno'
    texto = re.sub(r'\b[5-8]/10\b', ' good ', texto)

    # Detectar notas bajas como 1/10 a 4/10 y convertirlos en 'malo'
    texto = re.sub(r'\b[0-4]/10\b', ' bad ', texto)

    return texto


In [15]:
# Ejemplo: Acentos
texto = 'I went to a café in Québec'
print(f"Original: {texto}")
print(f"Transformado: {quitar_acentos(texto)}")

# Ejemplo: Contracciones
print("\nContracciones:")
texto = "I'm going to the store, I hope isn't closed"
print(f"Original: {texto}")
print(f"Transformado: {transformar_contracciones(texto)}")


# Ejemplo: Normalizar calificaciones
print("\nNormalizar calificaciones:")
texto = "The movie was a 10/10, but the ending was 2/10. Food was 6/10"
print(f"Original: {texto}")
print(f"Transformado: {normalizar_calificaciones(texto)}")

Original: I went to a café in Québec
Transformado: I went to a cafe in Quebec

Contracciones:
Original: I'm going to the store, I hope isn't closed
Transformado: I am going to the store, I hope is not closed

Normalizar calificaciones:
Original: The movie was a 10/10, but the ending was 2/10. Food was 6/10
Transformado: The movie was a  excelent , but the ending was  bad . Food was  good 


In [16]:
def clean_doc(doc):
  # transformar a minúsculas
  text = doc.lower()

  # normalizar calificaciones numéricas para que sean texto
  text = normalizar_calificaciones(text)

  # quitar acentos=, ej: `café` se convierte en `cafe`
  text = quitar_acentos(text)

  # expander contracciones, ej: `isn't` se transforma a `is not`
  text = transformar_contracciones(text)

  # Remplazamos lo que NO sea alfanuméricos por espacio, para que al hacer split
  # evitemos que se junten palabras como minutes.MAJOR, cosa que pasaría si
  # solo quitaramos los signos de puntuacion y otros caractéres no alfabéticos
  text = re.sub(r'[^a-z0-9]+', ' ', text)

  # separamos por espacio
  tokens = text.split()

  # quitamos tokens no alfabéticos
  tokens = [w for w in tokens if w.isalpha()]

  # Normaliza caracgteres repetidos
  tokens = [normalize_repeated_chars(token) for token in tokens]

  # Lemmatize
  tokens = [get_root_base(token) for token in tokens]

  # Eliminamos espacios en blanco, si es que aún existen
  tokens = [re.sub(r'\s+', ' ', token).strip() for token in tokens]

  # Dejamos los tokens con logitud mayor a 1 y que no sean stopwords
  tokens = [w for w in tokens if w not in stopwords_set and len(w) > 1]

  return tokens

In [17]:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


Xclean = [clean_doc(doc) for doc in X]


# *********** Aquí termina la sección de agregar código *************

In [18]:
# Despleguemos los primeros comentarios después de tu proceso de limpieza:

for x in Xclean[0:5]:
  print(x)


['no', 'way', 'plug', 'unless', 'go', 'converter']
['good', 'case', 'excellent', 'value']
['great', 'jawbone']
['tie', 'charger', 'conversation', 'last', 'minute', 'major', 'problem']
['mic', 'great']


# **Pregunta - 3:**



Realicemos una partición aleatoria con los porcentajes que consideres más adecuados. Utiliza una semilla para su reproducibilidad.

In [19]:

# ************* Inicia la sección de agregar código:*****************************


y_fix = y.values.ravel().astype(int);

Xtrain, x_val_and_test, ytrain, y_val_and_test = train_test_split(
    Xclean, y_fix,
    train_size=.70, shuffle=True, random_state=42, stratify=y_fix
)
Xval, Xtest, yval, ytest = train_test_split(
    x_val_and_test, y_val_and_test,
    test_size=.50, shuffle=True, random_state=42, stratify=y_val_and_test
)


# *********** Termina la sección de agregar código *************


# verificemos las dimensiones obtenidas:
print('X,y Train:', len(Xtrain), len(ytrain))
print('X,y Val:', len(Xval), len(yval))
print('X,y Test', len(Xtest), len(ytest))

X,y Train: 2100 2100
X,y Val: 450 450
X,y Test 450 450


# **Pregunta - 4:**




### **Construye tu vocabulario a continuación utilizando solamente el conjunto de Train:**


In [20]:
# a.	Usa el conjunto de entrenamiento para generar tu vocabulario
#     con un tamaño que consideres adecuado:


# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

mi_diccionario = Counter()

for k in range(len(Xtrain)):
    mi_diccionario.update(Xtrain[k])


min_freq = 3

mi_diccionario = Counter({
    w: f for w, f in mi_diccionario.items()
    if f >= min_freq
})


# *********** Aquí termina la sección de agregar código *************




In [21]:
# b.	Indica el tamaño del vocabulario generado.

print('Longitud del vocabulario generado:')


# ******* Inicia la sección de agregar código: ***********


print(f'Longitud del diccionario: {len(mi_diccionario)}')
print('\n(word,frequency):')
print(mi_diccionario.most_common(10))




# *********** Aquí termina la sección de agregar código *************

Longitud del vocabulario generado:
Longitud del diccionario: 934

(word,frequency):
[('not', 398), ('good', 201), ('great', 156), ('movie', 147), ('film', 124), ('phone', 123), ('bad', 121), ('one', 109), ('work', 104), ('like', 100)]


In [22]:
# c.	Con el vocabulario generado, filtra los conjuntos de entrenamiento,
#     validación y prueba para que todos los comentarios usen solamente las
#     palabras de este vocabulario.

#     Llamar train_X, val_X y test_X a estos tres conjuntos.


# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


train_X = []
for ss in Xtrain:
  train_X.append([w for w in ss if w in mi_diccionario])

val_X = []
for ss in Xval:
  val_X.append([w for w in ss if w in mi_diccionario])

test_X = []
for ss in Xtest:
  test_X.append([w for w in ss if w in mi_diccionario])


# *********** Aquí termina la sección de agregar código *************


In [23]:
# Vemos el resultado de los primeros comentarios del conjunto de validación:

for ss in val_X[0:5]:
  print(ss)

['long', 'bite']
['wind', 'completely', 'useless']
['great', 'character', 'actor']
['price', 'reasonable', 'flavor', 'spot', 'sauce', 'home', 'make', 'not']
['character', 'interest', 'want', 'find', 'long', 'movie', 'go', 'think', 'people', 'surprise', 'not', 'make']


In [24]:
for k in range(3):
  print('Antes:', Xtrain[k])
  print('Después:', train_X[k])

Antes: ['least', 'pas', 'order', 'food', 'arrive', 'not', 'busy']
Después: ['least', 'pas', 'order', 'food', 'arrive', 'not']
Antes: ['toro', 'tartare', 'cavier', 'extraordinary', 'like', 'thinly', 'slice', 'wagyu', 'white', 'truffle']
Después: ['like', 'slice', 'white']
Antes: ['use', 'colour', 'french', 'flag', 'three', 'film', 'nothing', 'short', 'incredible', 'every', 'shoot', 'every', 'scene', 'like', 'work', 'art']
Después: ['use', 'three', 'film', 'nothing', 'short', 'incredible', 'every', 'shoot', 'every', 'scene', 'like', 'work', 'art']


# **Pregunta - 5:**

Incluye tus comentarios sobre cada modelo de HuggingFace indicado.

### ++++++++ Inicia la sección de agregar texto: +++++++++++

* **a) bge-base-en-v1.5**

Este modelo es el "justo medio" ideal para proyectos escolares y profesionales estándar. Nos da un excelente balance entre velocidad de procesamiento y precisión semántica. Su gran ventaja es que genera vectores de 768 dimensiones, lo que significa que comprime el significado de las palabras en un tamaño muy cómodo para que algoritmos sencillos (como la Regresión Logística) aprendan rápido sin consumir toda la memoria RAM de nuestro entorno. Es el modelo que elegimos para el código porque es ligero, eficiente y muy potente para textos cortos como nuestras reseñas.

* **b) bge-large-en-v1.5**

Es el "hermano mayor" del modelo anterior. En lugar de 768, este modelo genera vectores más grandes, de 1024 dimensiones. Al tener más espacio numérico, es capaz de capturar sutilezas del lenguaje mucho más complejas, metáforas o contextos ocultos profundos. Sin embargo, este poder extra viene con un costo: requiere mucha más capacidad de cómputo y puede llegar a saturar la memoria si no se cuenta con una buena tarjeta gráfica. Para un dataset de 3,000 registros cortos como el nuestro, usarlo podría ser excesivo.

* **c)	e5-base-v2**

Este modelo utiliza una estrategia de entrenamiento muy interesante basada en "texto contrastivo" o sea que aprende comparando frases muy similares contra frases muy diferentes. Al igual que el modelo bge-base, genera vectores de 768 dimensiones. Es un competidor directo muy fuerte que especialmente cuando los textos tienen estructuras de preguntas y respuestas o cuando buscamos similitud pura entre documentos. Cualquiera de los dos (bge-base o e5-base) habría hecho un buen trabajo con nuestras reseñas de Amazon, IMDb y Yelp.

### ++++++++ Termina la sección de agregar texto: +++++++++++

# **Pregunta - 6:**

In [25]:
# a) Cargar el modelo de embeddings de HuggingFace seleccionado:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


model = SentenceTransformer('BAAI/bge-base-en-v1.5')


# *********** Aquí termina la sección de agregar código *************

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [26]:
# b) Primeros 3 elementos clave:valor del diccionario generado.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

model_name = model.tokenizer.name_or_path
embeddings_dict_file = f"{model_name.replace('/', '_')}.pkl"

try:
  # Cargando el diccionario si existe
  with open(embeddings_dict_file, "rb") as f:
    print(f"Cargando diccionario para: {embeddings_dict_file}")
    diccionario_embeddings = pickle.load(f)
except:
  print(f"Creando diccionario para: {embeddings_dict_file}")
  diccionario_embeddings = {
    palabra: model.encode(palabra).tolist() for palabra in mi_diccionario
  }

  # guardar el diccionario para evitar recalcularlo la siguiente ejecución
  with open(embeddings_dict_file, "wb") as f:
    pickle.dump(diccionario_embeddings, f)


# *********** Aquí termina la sección de agregar código *************



Creando diccionario para: BAAI_bge-base-en-v1.5.pkl


In [27]:
# Ejemplo: Imprimir la dimensión del vector de la palabra "data"

vocab_size = model.tokenizer.vocab_size
print(f"\nTamaño de vocabulario del modelo: {vocab_size}")
print(f"Dimensión de embeedings de modelo {model.model_card_data.model_name}: {model.get_embedding_dimension()}")


vector_data = diccionario_embeddings["data"]
print(f"Dimensión de la palabra 'data': {len(vector_data)}")

print(f'\nprimeros 3 elementos:')
elems = list(diccionario_embeddings.items())[:3]
print(elems[0])
print(elems[1])
print(elems[2])


Tamaño de vocabulario del modelo: 30522
Dimensión de embeedings de modelo None: 768
Dimensión de la palabra 'data': 768

primeros 3 elementos:
('least', [-0.07625748962163925, 0.0005580547149293125, 0.06365200132131577, -0.01665383391082287, 0.016517341136932373, 0.01978631503880024, 0.05199058726429939, -0.015039267018437386, -0.06755497306585312, -0.01925010047852993, 0.010020514018833637, 0.04400729015469551, -0.02320297434926033, 0.041109487414360046, -0.0065568676218390465, 0.018007373437285423, 0.018874390050768852, -0.008020740002393723, -0.0022472634445875883, -0.051779747009277344, 0.036640215665102005, -0.012372363358736038, -0.017210960388183594, -0.020777402445673943, 0.04184136912226677, 0.04671876132488251, 0.03416763246059418, -0.026272058486938477, -0.05630558356642723, 0.011588959954679012, 0.025440916419029236, 0.006163349375128746, 0.010629765689373016, -0.0008954282966442406, 0.040694672614336014, 0.02039279229938984, 0.02491174265742302, 0.029054539278149605, -0.0

In [28]:
# c) Tamaño del diccionario generado:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


print(
    f'tamaño: {len(diccionario_embeddings)}, '
    f'dimensión: {len(vector_data)}'
)


# *********** Aquí termina la sección de agregar código *************


tamaño: 934, dimensión: 768


# **Pregunta - 7:**




Generamos los vectores embebidos a partir de los conjuntos de entrenamiento, validación y prueba y con las características indicadas en el archivo PDF.

Los llamaremos trainEmb, valEmb y testEmb, respectivamente.


In [29]:
# a) Comentarios con vectores embebidos.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


trainEmb = [
    np.mean([diccionario_embeddings[token] for token in doc], axis=0) if doc
    else np.zeros(len(vector_data))
    for doc in train_X
]

valEmb = [
    np.mean([diccionario_embeddings[token] for token in doc], axis=0) if doc
    else np.zeros(len(vector_data))
    for doc in val_X
]

testEmb = [
    np.mean([diccionario_embeddings[token] for token in doc], axis=0) if doc
    else np.zeros(len(vector_data))
    for doc in test_X
]

# *********** Aquí termina la sección de agregar código *************

In [30]:
# b) Dimensiones de los conjuntos trainEmb, valEmb y testEmb.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

print(f'trainEmb: {np.array(trainEmb).shape}')
print(f'valEmb: {np.array(valEmb).shape}')
print(f'testEmb: {np.array(testEmb).shape}')


# *********** Aquí termina la sección de agregar código *************

trainEmb: (2100, 768)
valEmb: (450, 768)
testEmb: (450, 768)


# **Pregunta - 8:**

Debido a que trainEmb, valEmb y testEmb contienen un único vector promedio por comentario, el número de tokens se calculó utilizando los tokens presentes en los conjuntos procesados (train_X, val_X, test_X) empleados para generar dichos embeddings.


In [31]:
# Número de tokens generedos al obtener cada uno de los conjuntos trainEmb, valEmb y testEmb.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


num_tokens_train = sum(len(tokens) for tokens in train_X)
num_tokens_val = sum(len(tokens) for tokens in val_X)
num_tokens_test = sum(len(tokens) for tokens in test_X)

print(f'Número de tokens en trainEmb: {num_tokens_train}')
print(f'Número de tokens en valEmb: {num_tokens_val}')
print(f'Número de tokens en testEmb: {num_tokens_test}')

# *********** Aquí termina la sección de agregar código *************

Número de tokens en trainEmb: 10339
Número de tokens en valEmb: 2026
Número de tokens en testEmb: 1991


In [32]:
import random
random.seed(42)

def encontrar_palabra_mas_cercana(vector_objetivo, diccionario):
    mejor_palabra = None
    mayor_similitud = -1.0  # El rango de la similitud de coseno es de -1 a 1

    for palabra, vector_palabra in diccionario.items():
        # util.cos_sim calcula la similitud de coseno entre ambos vectores
        similitud = util.cos_sim(vector_objetivo, vector_palabra).item()

        if similitud > mayor_similitud:
            mayor_similitud = similitud
            mejor_palabra = palabra

    return mejor_palabra, mayor_similitud

def debug_document(idx):
  palabra_cercana, score = encontrar_palabra_mas_cercana(
      trainEmb[idx].astype(np.float32), diccionario_embeddings
  )
  print("Documento en posición:", train_X[idx])
  print(f"El vector promedio se parece más a la palabra: '{palabra_cercana}'")
  print(f"Score de similitud de coseno: {score:.4f}")
  print('-'*10)

# Get 5 random indexes of documents
num_indexes = 5
random_indexes = random.sample(range(len(train_X)), num_indexes)
for idx in random_indexes:
  debug_document(idx)



Documento en posición: ['poor', 'service']
El vector promedio se parece más a la palabra: 'poor'
Score de similitud de coseno: 0.8667
----------
Documento en posición: ['movie', 'show', 'lot', 'best', 'make', 'look', 'appeal']
El vector promedio se parece más a la palabra: 'show'
Score de similitud de coseno: 0.8508
----------
Documento en posición: ['terrible', 'management']
El vector promedio se parece más a la palabra: 'management'
Score de similitud de coseno: 0.8445
----------
Documento en posición: ['not', 'charge', 'cingular', 'phone']
El vector promedio se parece más a la palabra: 'phone'
Score de similitud de coseno: 0.8554
----------
Documento en posición: ['not', 'scene', 'like', 'previous', 'movie', 'terrible']
El vector promedio se parece más a la palabra: 'ago'
Score de similitud de coseno: 0.8217
----------


# **Pregunta - 9:**



Entrenamiento y reporte de los modelos de Regresión Logística y Bosque Aleatorio (Random Forest).


In [33]:
# 9a) REGRESIÓN LOGÍSTICA:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********
modeloLR = LogisticRegression(
  max_iter=1500,
  C=.01,
  penalty='l2',
  solver='liblinear',
  random_state=1
)
modeloLR.fit(trainEmb, ytrain)

y_val_pred = modeloLR.predict(valEmb)
print("Validation Set Performance:")
print(classification_report(yval, y_val_pred))

print('LR: Train-accuracy: %.2f%%' % (100*modeloLR.score(trainEmb, ytrain)))
print('LR: Val-accuracy: %.2f%%' % (100*modeloLR.score(valEmb, yval)))
print('LR: difference: %.2f%%' % (100*(modeloLR.score(trainEmb, ytrain)-modeloLR.score(valEmb, yval))))

# *********** Aquí termina la sección de agregar código *************


Validation Set Performance:
              precision    recall  f1-score   support

           0       0.84      0.80      0.82       225
           1       0.81      0.84      0.83       225

    accuracy                           0.82       450
   macro avg       0.82      0.82      0.82       450
weighted avg       0.82      0.82      0.82       450

LR: Train-accuracy: 82.76%
LR: Val-accuracy: 82.22%
LR: difference: 0.54%


In [34]:
# 9b) BOSQUE ALEATORIO (Random Forest):

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


modeloRF = RandomForestClassifier(
  n_estimators=150,
  max_depth=5,
  min_samples_leaf=1,
  min_samples_split=7,
  ccp_alpha=0.05, # 0.07
  random_state=1,
  n_jobs=-1
)
modeloRF.fit(trainEmb, ytrain)

y_val_pred = modeloRF.predict(valEmb)
print("Validation Set Performance:")
print(classification_report(yval, y_val_pred))

print('RF: Train-accuracy: %.2f%%' % (100*modeloRF.score(trainEmb, ytrain)))
print('RF: Val-accuracy: %.2f%%' % (100*modeloRF.score(valEmb, yval)))
print('RF: difference: %.2f%%' % (100*(modeloRF.score(trainEmb, ytrain)-modeloRF.score(valEmb, yval))))


# *********** Aquí termina la sección de agregar código *************

Validation Set Performance:
              precision    recall  f1-score   support

           0       0.81      0.70      0.75       225
           1       0.74      0.84      0.78       225

    accuracy                           0.77       450
   macro avg       0.77      0.77      0.77       450
weighted avg       0.77      0.77      0.77       450

RF: Train-accuracy: 79.67%
RF: Val-accuracy: 76.89%
RF: difference: 2.78%


In [35]:
#@title Mejor modelo usando promedio de embeddings

mejorModelo = modeloLR
y_test_pred = mejorModelo.predict(testEmb)
print("Validation Set Performance:")
print(classification_report(ytest, y_test_pred))


print('Train-accuracy: %.2f%%' % (100*mejorModelo.score(trainEmb, ytrain)))
print('Val-accuracy: %.2f%%' % (100*mejorModelo.score(testEmb, ytest)))
print('difference: %.2f%%' % (100*(mejorModelo.score(trainEmb, ytrain)-mejorModelo.score(testEmb, ytest))))


print('\nMatriz de confusión del mejor modelo en proporciones:')
print(confusion_matrix(ytest, y_test_pred, labels=[0,1]) / y_test_pred.shape[0])

Validation Set Performance:
              precision    recall  f1-score   support

           0       0.84      0.80      0.82       225
           1       0.81      0.84      0.83       225

    accuracy                           0.82       450
   macro avg       0.82      0.82      0.82       450
weighted avg       0.82      0.82      0.82       450

Train-accuracy: 82.76%
Val-accuracy: 82.44%
difference: 0.32%

Matriz de confusión del mejor modelo en proporciones:
[[0.40222222 0.09777778]
 [0.07777778 0.42222222]]


# **Pregunta - 10**

**Proceso basado en modelos Preentrenados**

In [36]:
Xnp = X.to_numpy()
ynp = y.to_numpy()

print(
    "X len:", len(Xnp), '\n'
    "y len:", len(ynp)
)

X len: 3000 
y len: 3000


In [37]:
# 10a) Partición.:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


X_train, x_val_and_test, y_train, y_val_and_test = train_test_split(
    Xnp, ynp,
    train_size=.70, shuffle=True, random_state=42, stratify=ynp
)
X_val, X_test, y_val, y_test = train_test_split(
    x_val_and_test, y_val_and_test,
    test_size=.50, shuffle=True, random_state=42, stratify=y_val_and_test
)

# verificemos las dimensiones obtenidas:
print('X,y Train:', len(X_train), len(y_train))
print('X,y Val:', len(X_val), len(y_val))
print('X,y Test', len(X_test), len(y_test))


# *********** Aquí termina la sección de agregar código *************

X,y Train: 2100 2100
X,y Val: 450 450
X,y Test 450 450


In [38]:
# 10b) Vectores embebidos:

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

trainEmb_2 = model.encode(X_train.tolist())
valEmb_2 = model.encode(X_val.tolist())
testEmb_2 = model.encode(X_test.tolist())


# *********** Aquí termina la sección de agregar código *************

In [39]:
print(f'trainEmb_2: {np.array(trainEmb_2).shape}')
print(f'valEmb_2: {np.array(valEmb_2).shape}')
print(f'testEmb_2: {np.array(testEmb_2).shape}')

trainEmb_2: (2100, 768)
valEmb_2: (450, 768)
testEmb_2: (450, 768)


In [40]:
# Conteo de tokens usados
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-base-en-v1.5")

def count_model_tokens(texts):
    return sum(
        len(tokenizer(text, truncation=True, max_length=512)["input_ids"])
        for text in texts
    )

num_tokens_train_2 = count_model_tokens(X_train)
num_tokens_val_2 = count_model_tokens(X_val)
num_tokens_test_2 = count_model_tokens(X_test)

print(f"Tokens utilizados para generar trainEmb_2: {num_tokens_train_2}")
print(f"Tokens utilizados para generar valEmb_2: {num_tokens_val_2}")
print(f"Tokens utilizados para generar testEmb_2: {num_tokens_test_2}")

Tokens utilizados para generar trainEmb_2: 36074
Tokens utilizados para generar valEmb_2: 7650
Tokens utilizados para generar testEmb_2: 7481


In [41]:
# 10c) REGRESIÓN LOGÍSTICA.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

modeloLR_2 = LogisticRegression(
  max_iter=1500,
  C=5,
  penalty='l2',
  solver='liblinear',
  random_state=1
)
modeloLR_2.fit(trainEmb_2, y_train)

y_val_pred = modeloLR_2.predict(valEmb_2)
print("Validation Set Performance:")
print(classification_report(y_val, y_val_pred))

print('LR: Train-accuracy: %.2f%%' % (100*modeloLR_2.score(trainEmb_2, y_train)))
print('LR: Val-accuracy: %.2f%%' % (100*modeloLR_2.score(valEmb_2, y_val)))
print('LR: difference: %.2f%%' % (100*(modeloLR_2.score(trainEmb_2, y_train)-modeloLR_2.score(valEmb_2, y_val))))

# *********** Aquí termina la sección de agregar código *************

Validation Set Performance:
              precision    recall  f1-score   support

           0       0.98      0.97      0.97       225
           1       0.97      0.98      0.97       225

    accuracy                           0.97       450
   macro avg       0.97      0.97      0.97       450
weighted avg       0.97      0.97      0.97       450

LR: Train-accuracy: 97.52%
LR: Val-accuracy: 97.33%
LR: difference: 0.19%


In [42]:
# 10d) BOSQUE ALEATORIO.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********

modeloRF_2 = RandomForestClassifier(
  n_estimators=100,
  max_depth=5,
  min_samples_leaf=1,
  min_samples_split=5,
  # ccp_alpha=0.2,
  random_state=1,
  n_jobs=-1
)
modeloRF_2.fit(trainEmb_2, y_train)

y_val_pred = modeloRF_2.predict(valEmb_2)
print("Validation Set Performance:")
print(classification_report(y_val, y_val_pred))

print('RF: Train-accuracy: %.2f%%' % (100*modeloRF_2.score(trainEmb_2, y_train)))
print('RF: Val-accuracy: %.2f%%' % (100*modeloRF_2.score(valEmb_2, y_val)))
print('RF: difference: %.2f%%' % (100*(modeloRF_2.score(trainEmb_2, y_train)-modeloRF_2.score(valEmb_2, y_val))))



# *********** Aquí termina la sección de agregar código *************

Validation Set Performance:
              precision    recall  f1-score   support

           0       0.98      0.96      0.97       225
           1       0.97      0.98      0.97       225

    accuracy                           0.97       450
   macro avg       0.97      0.97      0.97       450
weighted avg       0.97      0.97      0.97       450

RF: Train-accuracy: 97.38%
RF: Val-accuracy: 97.33%
RF: difference: 0.05%


# **Pregunta - 11:**

In [43]:
#@title Mejor modelo usando embeddings automáticos

# Reporte del mejor modelo y partición con el conjunto de Prueba.

# ******* Inlcuye a continuación todas las líneas de código y celdas que requieras: ***********


mejorModelo_2 = modeloLR_2
y_test_pred = mejorModelo_2.predict(testEmb_2)
print("Validation Set Performance:")
print(classification_report(y_test, y_test_pred))


print('Train-accuracy: %.2f%%' % (100*mejorModelo_2.score(trainEmb_2, y_train)))
print('Val-accuracy: %.2f%%' % (100*mejorModelo_2.score(testEmb_2, y_test)))
print('difference: %.2f%%' % abs(100*(mejorModelo_2.score(trainEmb_2, y_train)-mejorModelo_2.score(testEmb_2, y_test))))


print('\nMatriz de confusión del mejor modelo en proporciones:')
print(confusion_matrix(y_test, y_test_pred, labels=[0,1]) / y_test_pred.shape[0])


# *********** Aquí termina la sección de agregar código *************

Validation Set Performance:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98       225
           1       0.98      0.98      0.98       225

    accuracy                           0.98       450
   macro avg       0.98      0.98      0.98       450
weighted avg       0.98      0.98      0.98       450

Train-accuracy: 97.52%
Val-accuracy: 97.78%
difference: 0.25%

Matriz de confusión del mejor modelo en proporciones:
[[0.48888889 0.01111111]
 [0.01111111 0.48888889]]


# **Pregunta - 12:**



Incluye tus comentarios finales de la actividad.

### ++++++++ Inicia la sección de agregar texto: +++++++++++
Comentarios finales de la actividad:

1. Una de las observaciones más contundentes al evaluar los modelos entrenados con embeddings (como LogisticRegression y RandomForestClassifier) es la reducción de la brecha de accuracy entre el conjunto de entrenamiento (Xtrain) y el de validación (Xval). A diferencia de las matrices dispersas de conteo, la representación densa de 768 dimensiones funciona como un regularizador implícito de características, forzando a los algoritmos a generalizar mucho mejor ante datos no observados.

2. Los modelos tradicionales de conteo son ciegos a la sinonimia; si una palabra clave cambia por un sinónimo en el conjunto de prueba, el clasificador pierde dicha señal. Al utilizar BAAI/bge-base-en-v1.5, el texto se mapea en un espacio geométrico continuo donde palabras con cargas semánticas similares se ubican muy cerca unas de otras. Esto le permite al clasificador inferir correctamente la polaridad positiva o negativa de una reseña incluso si los tokens exactos no estuvieron presentes de forma repetida en el conjunto de entrenamiento.

3. La estrategia de reducción mediante el promedio de vectores por documento demostró ser una solución muy eficiente. Condensó documentos de longitudes variables provenientes de tres dominios completamente heterogéneos en matrices numéricas compactas y homogéneas de tamaño fijo ($2100 \times 768$ para Train, y $450 \times 768$ para Validación y Prueba).

4. La transferencia de conocimiento a través de los embeddings preentrenados de HuggingFace da al pipeline una robustez superior para clasificar sentimientos en un dataset mixto. Al no depender exclusivamente de la frecuencia estadística local de nuestro corpus, el modelo se beneficia del entendimiento masivo de la sintaxis del idioma inglés con el que fue preentrenado el modelo base, logrando un desempeño predictivo sólido y estable en el conjunto final de prueba.
### ++++++++ Termina la sección de agregar texto: +++++++++++

# **Fin de la Actividad de Vectores Embebidos - HuggingFace**